# 05 · Estudio de ablación (periodo común 2025–2026)
Pedido del especialista (5.4): 1) modelo base → 2) + codificación cíclica → 3) + Weighted Focal Loss; se extiende con precursores locales y globales para responder **¿aporta ROTI?**. Misma partición, mismo LOOKBACK/HORIZON, misma arquitectura, 3 semillas, evaluación en **Validación**. Test se usa solo en el notebook 09.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, variante, ruta
from s4.pipeline import preparar_datos, correr_variante, baselines, construir_particiones, evaluar_y_registrar
from s4.registry import leer, resumir_semillas
from s4.train import configurar_gpu
from s4 import features as ft
configurar_gpu()
cfg = cargar_config('../config/ablacion_2025_2026.yaml')
datos = preparar_datos(cfg)

In [ ]:
# Baselines en validación (referencia de todas las filas)
parts = construir_particiones(cfg, datos, ft.CONJUNTOS["A_S4"])
for nombre, yhat in baselines(cfg, datos, parts, "val").items():
    evaluar_y_registrar(cfg, parts["val"], yhat, nombre, "ablacion")

In [ ]:
VARIANTES = [
    ("A · S4 + MSE",                   "A_S4",         {"perdida.tipo": "mse"}),
    ("B · + cíclica + MSE",            "B_S4_ciclica", {"perdida.tipo": "mse"}),
    ("C · + cíclica + WFL",            "B_S4_ciclica", {}),
    ("D · + TEC/ROTEC/ROTI + WFL",     "D_locales",    {}),
    ("D' · + solo ROTI + WFL",         "D_solo_ROTI",  {}),
    ("E · + índices globales + WFL",   "E_completo",   {}),
]
for nombre, conj, ov in VARIANTES:
    print("\n==", nombre)
    correr_variante(variante(cfg, **ov), datos, ft.CONJUNTOS[conj], nombre, "ablacion")

In [ ]:
reg = leer(ruta(cfg, "experimentos") / f"registro_{cfg['nombre']}.csv", grupo="ablacion", particion="val")
tabla = resumir_semillas(reg, por=("modelo",), metricas=("RMSE_global","RMSE_evento","RMSE_calma","POD","Precision","HSS","POD_inicio"))
tabla.to_csv(ruta(cfg, "reportes") / "tables/tabla_ablacion.csv", index=False); tabla.round(4)